In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from boruta import BorutaPy
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


In [73]:
df = pd.read_excel('./data.xlsx')

column_mapping = {
    '序号': 'id',
    '姓名': 'name',
    '年龄（岁）': 'age',
    '性别（男=1；女=0）': 'gender',
    'BMI（kg/m^2）': 'bmi',
    '是否高血压（是=1；否=0）': 'hypertension',
    '是否心脏病（是=1；否=0）': 'heart_disease',
    '是否糖尿病（是=1；否=0）': 'diabetes',
    '是否慢性咽炎（是=1；否=0）': 'chronic_pharyngitis',
    '是否慢性鼻炎（是=1；否=0）': 'chronic_rhinitis',
    '用嗓习惯（无特殊用嗓习惯=0&#10;过度用嗓（如频繁高声说话、唱歌等）=1）': 'voice_habit',
    '饮食（饮食习惯规律=0&#10;经常食用刺激性饮食（如辛辣、油腻食物）=1）': 'diet',
    '作息（作息规律=0&#10;作息不规律（如经常熬夜）=1）': 'schedule',
    '吸烟(有吸烟习惯=1;无=0)': 'smoking',
    '饮酒(有饮酒习惯=1;无=0)': 'drinking',
    '职业类型（普通职业=0；用嗓频率高如教师、销售等职业=1）': 'occupation',
    'TSH（mIU/L）': 'tsh',
    'T3（nmol/L）': 't3',
    'T4（nmol/L）': 't4',
    'FT3（pmol/L）': 'ft3',
    'FT4（pmol/L）': 'ft4',
    '是否桥本甲状腺炎（是=1；否=0）': 'hashimoto',
    '超声下结节是否靠背侧（是=1；否=0）': 'nodule_posterior',
    '超声下结节是否靠气管（是=1；否=0）': 'nodule_trachea',
    '超声下结节最大径（cm）': 'nodule_size',
    '是否联合事件（是=1；否=0）': 'combined_event',
    '是否LOS（是=1；否=0）': 'los',
    '优势侧V1振幅（μV）': 'v1_amplitude',
    '优势侧R1振幅（μV）': 'r1_amplitude',
    '优势侧V振幅下降率（%）': 'v_amplitude_drop',
    '优势侧R振幅下降率（%）': 'r_amplitude_drop',
    '手术范围（单叶=1；双叶=2）': 'surgery_scope',
    '是否进行中央组淋巴结清扫术（是=1；否=0）': 'central_lymph_node',
    '是否进行侧颈淋巴结清扫术（否=0；单侧=1；双侧=2）': 'lateral_lymph_node',
    '手术方式（开放=0；腔镜=1）': 'surgery_method',
    '肌松（顺式阿曲库铵）剂量（mg）': 'muscle_relaxant_dose',
    '手术时间（min）': 'surgery_time',
    '病灶数量（单灶=1，多灶=2）': 'lesion_count',
    '嗓音障碍（是=1；否=0）': 'voice_disorder'
}

df.rename(columns=column_mapping, inplace=True)
print(df.columns)

# 定义特征矩阵 X 和标签 y
feature_columns = [col for col in df.columns if col not in ['id', 'name', 'voice_disorder']]
X_raw = df[feature_columns]
y = df['voice_disorder']

# 填充缺失值并标准化
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X = scaler.fit_transform(X_imputed)

#嗓音障碍特征
print(y.value_counts())


Index(['id', 'name', 'age', 'gender', 'bmi', 'hypertension', 'heart_disease',
       'diabetes', 'chronic_pharyngitis', 'chronic_rhinitis', 'voice_habit',
       'diet', 'schedule', 'smoking', 'drinking', 'occupation', 'tsh', 't3',
       't4', 'ft3', 'ft4', 'hashimoto', 'nodule_posterior', 'nodule_trachea',
       'nodule_size', 'combined_event', 'los', 'v1_amplitude', 'r1_amplitude',
       'v_amplitude_drop', 'r_amplitude_drop', 'surgery_scope',
       'central_lymph_node', 'lateral_lymph_node', 'surgery_method',
       'muscle_relaxant_dose', 'surgery_time', 'lesion_count',
       'voice_disorder'],
      dtype='object')
voice_disorder
0    148
1    101
Name: count, dtype: int64


In [74]:
def calculate_vif(X_df):
    vif_values = []
    for feature in X_df.columns:
        X_other = X_df.drop(columns=[feature]).values
        y_feature = X_df[feature].values
        model = LinearRegression()
        model.fit(X_other, y_feature)
        r2 = model.score(X_other, y_feature)
        vif = np.inf if np.isclose(r2, 1.0) else 1.0 / (1.0 - r2)
        tol = 0.0 if np.isinf(vif) else 1.0 / vif
        vif_values.append({'feature': feature, 'vif': vif, 'tolerance': tol})
    return pd.DataFrame(vif_values).sort_values('vif', ascending=False)


def feature_collinearity_selection(X, feature_names, vif_threshold=5.0, tol_threshold=0.1, rounds=2):
    X_df = pd.DataFrame(X, columns=feature_names)
    selected = list(feature_names)
    for round_idx in range(1, rounds + 1):
        vif_df = calculate_vif(X_df[selected])
        high_vif = vif_df[(vif_df['vif'] > vif_threshold) | (vif_df['tolerance'] < tol_threshold)]
        if high_vif.empty:
            print(f"第{round_idx}轮共线性分析：没有发现高 VIF 特征，停止剔除。")
            break
        drop_features = high_vif['feature'].tolist()
        selected = [f for f in selected if f not in drop_features]
        print(f"第{round_idx}轮共线性分析：删除特征 {drop_features}")
        if not selected:
            break
    return selected


def select_features(X, y, k=15):
    core_features = [
        'v1_amplitude', 'r1_amplitude', 'v_amplitude_drop', 'r_amplitude_drop', # 神经监测
        'surgery_scope', 'central_lymph_node', 'lateral_lymph_node', # 手术范围
        'surgery_time', 'nodule_size', 'nodule_posterior', 'nodule_trachea' # 结节与手术难度
    ]
    
    feature_names = [col for col in df.columns if col not in ['id', 'name', 'voice_disorder']]
    selected_collinearity = feature_collinearity_selection(X, feature_names)
    print(f"共线性筛选后特征数量: {len(selected_collinearity)}")

    n_select = min(k, len(selected_collinearity))
    rfe_estimator = LogisticRegression(random_state=42, max_iter=1000)
    rfe = RFE(estimator=rfe_estimator, n_features_to_select=n_select, step=1)
    rfe.fit(pd.DataFrame(X, columns=feature_names)[selected_collinearity], y)
    selected_rfe = [f for f, support in zip(selected_collinearity, rfe.support_) if support]
    print(f"RFE筛选后特征数量: {len(selected_rfe)}")

    # 强制保留核心特征
    for cf in core_features:
        if cf in feature_names and cf not in selected_rfe:
            selected_rfe.append(cf)
    print(f"强制保留核心特征后，最终特征数量: {len(selected_rfe)}")
    print(f"最终特征: {selected_rfe}") 
    
    X_selected = pd.DataFrame(X, columns=feature_names)[selected_rfe].values
    selected_indices = [feature_names.index(f) for f in selected_rfe]
    return X_selected, selected_indices, selected_rfe

X_selected, selected_indices, selected_features = select_features(X, y)

第1轮共线性分析：删除特征 ['r_amplitude_drop', 'v_amplitude_drop']
第2轮共线性分析：没有发现高 VIF 特征，停止剔除。
共线性筛选后特征数量: 34
RFE筛选后特征数量: 15
强制保留核心特征后，最终特征数量: 23
最终特征: ['bmi', 'voice_habit', 'occupation', 't3', 't4', 'ft3', 'ft4', 'hashimoto', 'nodule_posterior', 'nodule_trachea', 'nodule_size', 'combined_event', 'los', 'surgery_method', 'muscle_relaxant_dose', 'v1_amplitude', 'r1_amplitude', 'v_amplitude_drop', 'r_amplitude_drop', 'surgery_scope', 'central_lymph_node', 'lateral_lymph_node', 'surgery_time']


In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y)


In [82]:
from sklearn.base import BaseEstimator, ClassifierMixin

class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.model = PLSRegression(n_components=n_components)

    def fit(self, X, y):
        self.model = PLSRegression(n_components=self.n_components)
        self.model.fit(X, y)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        y_pred = self.model.predict(X).ravel()
        return (y_pred >= 0.5).astype(int)

    def predict_proba(self, X):
        y_pred = self.model.predict(X).ravel()
        y_pred = np.clip(y_pred, 0, 1)
        return np.vstack([1 - y_pred, y_pred]).T

    def get_params(self, deep=True):
        return {'n_components': self.n_components}

    def set_params(self, **params):
        if 'n_components' in params:
            self.n_components = params['n_components']
            self.model = PLSRegression(n_components=self.n_components)
        return self

class BorutaClassifier:
    def __init__(self, estimator=None, random_state=42, verbose=0):
        self.estimator = estimator if estimator is not None else RandomForestClassifier(n_estimators=100, random_state=random_state)
        self.random_state = random_state
        self.verbose = verbose
        self.support_ = None

    def fit(self, X, y):
        if BorutaPy is None:
            raise ImportError('BorutaPy is not installed. Please install boruta or boruta_py.')
        selector = BorutaPy(self.estimator, n_estimators='auto', random_state=self.random_state, verbose=self.verbose)
        selector.fit(X, y)
        self.support_ = selector.support_
        self.estimator.fit(X[:, self.support_], y)
        return self

    def predict(self, X):
        return self.estimator.predict(X[:, self.support_])

    def predict_proba(self, X):
        return self.estimator.predict_proba(X[:, self.support_])

models = {
    'DT': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(C=0.1, gamma='scale', kernel='linear', probability=True, random_state=42),
    'RF': RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_split=10, random_state=42),
    'GBDT': GradientBoostingClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=50, max_depth=5, learning_rate=0.01, subsample=0.7, use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=7, learning_rate=0.01, num_leaves=15, random_state=42, verbose=-1),
    'Logit': LogisticRegression(C=0.01, penalty='l2', random_state=42, max_iter=1000, solver='liblinear'),
    'BPNN': MLPClassifier(hidden_layer_sizes=(50,), alpha=0.0001, learning_rate_init=0.001, max_iter=500, random_state=42),
    'GNB': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'PLSDA': PLSDA(n_components=2),
    'Boruta': BorutaClassifier(random_state=42)
}       
from sklearn.ensemble import VotingClassifier

# 选择表现较好的模型进行集成
selected_models = [
    ('GBDT', models['GBDT']),
    ('RF', models['RF']),
    ('XGBoost', models['XGBoost']),
    ('SVM', models['SVM']),
    ('Logit', models['Logit'])
]

# 创建软投票集成模型
ensemble_model = VotingClassifier(estimators=selected_models, voting='soft')

# 添加到models字典
models['Ensemble'] = ensemble_model


In [83]:
def evaluate_models_cv(models, X, y, cv=5):
    results = {}
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    for name, model in models.items():
        accuracies = []
        aucs = []
        f1s = []
        sensitivities = []
        specificities = []
        ppvs = []
        npvs = []
        all_y_true = []
        all_y_pred = []
        all_y_pred_proba = []
        
        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]
            
            # 训练折中使用 SMOTE 进行过采样
            smote = SMOTE(random_state=42)
            X_train_res, y_train_res = smote.fit_resample(X_train_fold, y_train_fold)
            
            # 训练模型
            model.fit(X_train_res, y_train_res)
            # 预测
            y_pred = model.predict(X_test_fold)
            y_pred_proba = model.predict_proba(X_test_fold)[:, 1]
            # 评估指标
            accuracy = accuracy_score(y_test_fold, y_pred)
            auc_score = roc_auc_score(y_test_fold, y_pred_proba)
            f1 = f1_score(y_test_fold, y_pred, zero_division=0)
            tn, fp, fn, tp = confusion_matrix(y_test_fold, y_pred).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
            
            accuracies.append(accuracy)
            aucs.append(auc_score)
            f1s.append(f1)
            sensitivities.append(sensitivity)
            specificities.append(specificity)
            ppvs.append(ppv)
            npvs.append(npv)
            
            all_y_true.extend(y_test_fold)
            all_y_pred.extend(y_pred)
            all_y_pred_proba.extend(y_pred_proba)
        
        # 计算平均值
        avg_accuracy = np.mean(accuracies)
        avg_auc = np.mean(aucs)
        avg_f1 = np.mean(f1s)
        avg_sensitivity = np.mean(sensitivities)
        avg_specificity = np.mean(specificities)
        avg_ppv = np.mean(ppvs)
        avg_npv = np.mean(npvs)
        
        # 整体分类报告（基于所有预测）
        report = classification_report(all_y_true, all_y_pred)
        
        results[name] = {
            'model': model,
            'accuracy': avg_accuracy,
            'auc': avg_auc,
            'f1': avg_f1,
            'sensitivity': avg_sensitivity,
            'specificity': avg_specificity,
            'ppv': avg_ppv,
            'npv': avg_npv,
            'classification_report': report,
            'y_true_all': all_y_true,
            'y_pred_all': all_y_pred,
            'y_pred_proba_all': all_y_pred_proba
        }
        print(f"\n=== {name} Results (5-fold CV) ===")
        print(f"Average Accuracy: {avg_accuracy:.4f}")
        print(f"Average AUC: {avg_auc:.4f}")
        print(f"Average F1 score: {avg_f1:.4f}")
        print(f"Average Sensitivity (召回率): {avg_sensitivity:.4f}")
        print(f"Average Specificity: {avg_specificity:.4f}")
        print(f"Average PPV (阳性预测值): {avg_ppv:.4f}")
        print(f"Average NPV (阴性预测值): {avg_npv:.4f}")
        print(report)
    return results

results = evaluate_models_cv(models, X_selected, y)


=== DT Results (5-fold CV) ===
Average Accuracy: 0.6467
Average AUC: 0.6255
Average F1 score: 0.5348
Average Sensitivity (召回率): 0.5148
Average Specificity: 0.7363
Average PPV (阳性预测值): 0.5651
Average NPV (阴性预测值): 0.6938
              precision    recall  f1-score   support

           0       0.69      0.74      0.71       148
           1       0.57      0.51      0.54       101

    accuracy                           0.65       249
   macro avg       0.63      0.63      0.63       249
weighted avg       0.64      0.65      0.64       249


=== SVM Results (5-fold CV) ===
Average Accuracy: 0.7390
Average AUC: 0.7764
Average F1 score: 0.6684
Average Sensitivity (召回率): 0.6624
Average Specificity: 0.7899
Average PPV (阳性预测值): 0.6890
Average NPV (阴性预测值): 0.7796
              precision    recall  f1-score   support

           0       0.77      0.79      0.78       148
           1       0.68      0.66      0.67       101

    accuracy                           0.74       249
   macro avg  

In [84]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    'SVM': {
        'estimator': SVC(probability=True, random_state=42),
        'param_grid': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto']
        }
    },
    'XGBoost': {
        'estimator': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.7, 1.0]
        }
    },
    'LightGBM': {
        'estimator': LGBMClassifier(random_state=42, verbose=-1),
        'param_grid': {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1],
            'num_leaves': [15, 31, 63]
        }
    },
    'RF': {
        'estimator': RandomForestClassifier(random_state=42),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    },
    'GBDT': {
        'estimator': GradientBoostingClassifier(random_state=42),
        'param_grid': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5, 7]
        }
    },
    'Logit': {
        'estimator': LogisticRegression(random_state=42, max_iter=1000, solver='liblinear'),
        'param_grid': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'BPNN': {
        'estimator': MLPClassifier(random_state=42, max_iter=500),
        'param_grid': {
            'hidden_layer_sizes': [(50,), (100,), (100, 50)],
            'alpha': [0.0001, 0.001],
            'learning_rate_init': [0.001, 0.01]
        }
    },
    'KNN': {
        'estimator': KNeighborsClassifier(),
        'param_grid': {
            'n_neighbors': [3, 5, 7],
            'weights': ['uniform', 'distance']
        }
    },
    'PLSDA': {
        'estimator': PLSDA(),
        'param_grid': {
            'n_components': [1, 2, 3, 4, 5]
        }
    }
}


def tune_models(param_grids, X_train, y_train, cv=5):
    best_models = {}
    for name, config in param_grids.items():
        print(f"\nTuning {name}...")
        grid_search = GridSearchCV(
            config['estimator'],
            config['param_grid'],
            cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
            scoring='roc_auc',
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        best_models[name] = grid_search.best_estimator_
        print(f"{name} best params: {grid_search.best_params_}")
        print(f"{name} best CV AUC: {grid_search.best_score_:.4f}")
    return best_models

best_models = tune_models(param_grids, X_train, y_train)


Tuning SVM...
SVM best params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
SVM best CV AUC: 0.7944

Tuning XGBoost...
XGBoost best params: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 50, 'subsample': 0.7}
XGBoost best CV AUC: 0.8036

Tuning LightGBM...
LightGBM best params: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 100, 'num_leaves': 15}
LightGBM best CV AUC: 0.7905

Tuning RF...
RF best params: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 200}
RF best CV AUC: 0.8046

Tuning GBDT...
GBDT best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
GBDT best CV AUC: 0.8140

Tuning Logit...
Logit best params: {'C': 0.01, 'penalty': 'l2'}
Logit best CV AUC: 0.7977

Tuning BPNN...
BPNN best params: {'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001}
BPNN best CV AUC: 0.7208

Tuning KNN...
KNN best params: {'n_neighbors': 7, 'weights': 'distance'}
KNN best CV AUC: 0.7160

Tuning PLSDA...
PLSDA best params: 